In [1]:
import os
import numpy as np
import pandas as pd
import chart_studio as py
import plotly.figure_factory as ff
import plotly.graph_objs as go

from bisect import bisect_left
from numpy import pi as PI
from sqlalchemy import create_engine
from graph_utils import order_nodes, sort_graph_dataframe
from pycirclizely.parser import Matrix
from plotly.offline import iplot

In [2]:
DATABASE_PATH = "database/lattes.db"

In [3]:
engine = create_engine(f"sqlite:///{DATABASE_PATH}")
collaborations_query = """
WITH article_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        a.title AS collaboration,
        'artigo' AS type,
        a.year AS start,
        a.year AS end
    FROM authorship au1
    JOIN authorship au2 ON au1.article_id = au2.article_id AND au1.author_id < au2.author_id
    JOIN researchers r1 ON au1.author_id = r1.lattes_id
    JOIN researchers r2 ON au2.author_id = r2.lattes_id
    JOIN articles a ON au1.article_id = a.id
),
project_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        p.name AS collaboration,
        'projeto' AS type,
        p.start AS start,
        CAST(COALESCE(p.end, strftime('%Y', 'now')) AS INTEGER) AS end
    FROM participation p1
    JOIN participation p2 ON p1.project_id = p2.project_id AND p1.participant_id < p2.participant_id
    JOIN researchers r1 ON p1.participant_id = r1.lattes_id
    JOIN researchers r2 ON p2.participant_id = r2.lattes_id
    JOIN projects p ON p1.project_id = p.id
)
SELECT * FROM article_collaborations
UNION ALL
SELECT * FROM project_collaborations
"""

# Load dataframe with data extracted from the database query
try:
    collaborations: pd.DataFrame = pd.read_sql_query(collaborations_query, engine)
    print("✓ Dados de colaboração carregados com sucesso")
    print(f"  - {len(collaborations)} registros carregados")
except Exception as e:
    print(f"✗ Erro ao carregar dados de colaboração do banco de dados: {e}")
    print("  - Verifique se o banco de dados existe e contém as tabelas necessárias")
    raise


✓ Dados de colaboração carregados com sucesso
  - 807 registros carregados


In [4]:
collaboration_graph: pd.DataFrame = (
    collaborations.groupby(["researcher_1", "researcher_2"])
    .size()
    .reset_index(name="collaborations")
)

In [5]:
collaboration_graph = sort_graph_dataframe(
    collaboration_graph,
    "researcher_1",
    "researcher_2",
    "collaborations",
    order_nodes(collaboration_graph),
)

In [6]:
# Step 1: Create sorted list of all researchers
all_researchers = sorted(set(collaboration_graph['researcher_1']).union(set(collaboration_graph['researcher_2'])))

# Step 2: Create n x n matrix filled with zeros
n = len(all_researchers)
collab_matrix = np.zeros((n, n), dtype=int)

# Create a mapping from researcher name to index
researcher_to_index = {name: idx for idx, name in enumerate(all_researchers)}

# Step 3: Fill the matrix by iterating through the dataframe (more efficient)
for _, row in collaboration_graph.iterrows():
    i = researcher_to_index[row['researcher_1']]
    j = researcher_to_index[row['researcher_2']]
    collab_matrix[i, j] = collab_matrix[j, i] = row['collaborations']

print(collab_matrix.shape)
print(collab_matrix)

(105, 105)
[[ 0  0  0 ...  0  0  0]
 [ 0  0  0 ... 18  0  0]
 [ 0  0  0 ...  0  0  0]
 ...
 [ 0 18  0 ...  0  0  0]
 [ 0  0  0 ...  0  0  0]
 [ 0  0  0 ...  0  0  0]]


In [7]:
def moduloAB(angle: float, a: float = 0, b: float = 2 * PI) -> float:
    '''
    normalizes an angle to fit onto the unit circle identified with the interval [a, b). tipically b - a = 2π
    '''
    if a >= b:
        raise ValueError(f'Intervalos da faixa incorretos (a = {a}, b = {b}), forneça valores tais que a < b')
    y = (x - a) % (b - a)
    return y + b if y < 0 else y + a

def test_2PI(x: float) -> bool:
    return 0 <= x < 2 * PI

In [8]:
row_sums = 1
gap = 2* PI * 0.005 # gap between ideograms
ideogram_length = 2 * PI * np.asarray(row_sums)/ sum(row_sums) - gap * np.ones(n)
print(row_sums)

[ 16.  53.  13.  95.  29.   7.   1.  13.   5.   4.  24.  26.   1.   4.
  85.   1.  13.  14.  18.   4.   2.   1.  15.   9.  12.   4.   1.   4.
   4.  35.  11.  14.  10.   4.   1.  18.  22.   4.   1.  16.  46.   8.
 111.  11.   3.   7.   2.   4.  27.  18.  37.  29.  12.  28.  12.   6.
  35.   5.   3.   8.  27.   5.  17.   6.  22.   1.   9.   7.  40.  15.
   4.  29.   4.   6.  11.  12.   3.  12.   4.  30.   1.   2.   4.   7.
   1.   3.   8.  32.   7.  17.  21.   3.   6.   1.   9.  30.  46.   1.
   7.  33.   4.   5.  72.   2.   2.]


In [9]:
def get_ideogram_ends(ideogram_len: float, gap: float) -> tuple[float, float]:
    ideo_ends = []
    left = np.float64(0)
    for k in range(len(ideogram_len)):
        right = left + ideogram_len[k]
        ideo_ends.append((left, right))
        left = right + gap
    return ideo_ends

ideo_ends = get_ideogram_ends(ideogram_length, gap)

In [10]:
def make_ideogram_arc(R: float, phi: list[float], a: int = 50) -> float:
    if not test_2PI(phi[0]) or not test_2PI(phi[1]):
        phi = [moduloAB(t, 0, 2 * PI) for t in phi]
    length = (phi[1] - phi[0]) % 2*PI
    nr = 5 if length <= PI / 4 else int(a * length / PI)

    if phi[0] >= phi[1]:
        phi = [moduloAB(t, - PI, PI) for t in phi]
    theta = np.linspace(phi[0], phi[1], nr)
    return R * np.exp(1j * theta) # INFO: 1j is the notation for the imaginary unit √-1

In [11]:
# Set the ideograms labels colors from a gradation ranging from deep blue
# to bright yellow, passing through magenta. source:
# https://www.learnui.design/tools/data-color-picker.html

def hex_to_rgba(hex_color, opacity=0.75) -> str:
    """Convert hex color to RGBA with the specified opacity"""

    hex_color = hex_color.lstrip('#')
    RGB = [int(hex_color[i:i + 2], 16) for i in [0, 2, 4]]
    return f'rgba({RGB[0]}, {RGB[1]}, {RGB[2]}, {opacity})'

color_pallete = [
    '#003f5c', '#2f4b7c', '#665191', '#a05195',
    '#d45087', '#f95d6a', '#ff7c43', '#ffa600'
]

rgba_colors = [hex_to_rgba(color, 0.75) for color in color_pallete]

max_value = np.max(row_sums)
min_value = np.min(row_sums)
increment = np.ptp(row_sums) // len(color_pallete)
ranges = [ i * increment for i in range(len(color_pallete) - 1) ]
label_colors = [
    rgba_colors[bisect_left(ranges, row_sum)] for row_sum in row_sums
]
print(label_colors)

['rgba(102, 81, 145, 0.75)', 'rgba(249, 93, 106, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(255, 166, 0, 0.75)', 'rgba(160, 81, 149, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(102, 81, 145, 0.75)', 'rgba(102, 81, 145, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(255, 166, 0, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(102, 81, 145, 0.75)', 'rgba(102, 81, 145, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(102, 81, 145, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(160, 81, 149, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(102, 81, 145, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(47, 75, 124, 0.75)', 'rgba(102, 81, 145, 0.75)', 'rgba(102, 81, 1

In [12]:
def map_data(
    data_matrix: np.array, row_value: np.array, ideogram_length: np.array
) -> np.array:
    mapped=np.zeros(data_matrix.shape)
    for j in range(data_matrix.shape[0]):
        mapped[:, j] = ideogram_length * data_matrix[:,j] / row_value
    return mapped

mapped_data = map_data(collab_matrix, row_sums, ideogram_length)
idx_sort = np.argsort(mapped_data, axis=1)

In [13]:
# Compute the ribbon ends and store them as tuples in a list of lists

def make_ribbon_ends(mapped_data: np.array, ideo_ends: np.array, idx_sort: np.array):
    length = mapped_data.shape[0]
    ribbon_boundary=np.zeros((length, length + 1))
    ribbon_ends = []
    for k in range(length):
        ideogram_ribbons = []
        start=ideo_ends[k][0]
        ribbon_boundary[k][0] = start
        for j in range(1, length + 1):
            J = idx_sort[k][j - 1]
            ribbon_boundary[k][j] = start + mapped_data[k][J]
            ideogram_ribbons.append(
                (ribbon_boundary[k][j - 1], ribbon_boundary[k][j])
            )
            start = ribbon_boundary[k][j]
        ribbon_ends.append(ideogram_ribbons)
    return ribbon_ends
    
ribbon_ends = make_ribbon_ends(mapped_data, ideo_ends, idx_sort)

In [14]:
def control_pts(angle, radius):
    # angle: a 3-list containing angular coordinates of the control points b0, b1, b2
    # radius: the distance from b1 to the origin O(0,0) of the polar system of coordinates

    if len(angle) != 3:
        raise InvalidInputError('Angulo deve ser uma tupla de comprimento 3')
    # b_cplx represents the coordinates of the point B (the end point of a ribbon
    # segment) represented using complex numbers. This is an elegant mathematical
    # property that represents x and y polar coordinates in one variable, in the
    # real and imaginary parts of the number respectively.
    b_cplx=np.array([np.exp(1j*angle[k]) for k in range(3)])
    b_cplx[1] = radius * b_cplx[1]
    return zip(b_cplx.real, b.cplx.imag)

In [15]:
def ctrl_rib_chords(
    left_arc: tuple[float, float],
    right_arc: tuple[float, float], 
    radius: float
) -> list[list[complex]]:
    """
    Calculate control points for the two quadratic Bezier curves that form 
    the opposite sides of a ribbon connecting two arcs.
    
    The function generates control polygons for the top and bottom edges
    of a ribbon that connects two circular arcs. Each Bezier curve is defined
    by three control points derived from angular coordinates.
    
    Parameters
    ----------
    left_arc : Tuple[float, float]
        Angular coordinates (in radians) of the ribbon's starting arc ends.
        Format: (start_angle, end_angle) for the left-side arc.
    right_arc : Tuple[float, float]
        Angular coordinates (in radians) of the ribbon's ending arc ends.  
        Format: (start_angle, end_angle) for the right-side arc.
    radius : float
        The radius of the circle on which the control points are positioned.
        This is a common parameter for both control polygons.
    
    Returns
    -------
    List[List[complex]]
        A list containing two elements, each representing a control polygon:
        - First element: Control points for the top edge Bezier curve
        - Second element: Control points for the bottom edge Bezier curve
        Each control polygon is a list of complex numbers representing 
        Cartesian coordinates (x + iy) of the control points.
    
    Raises
    ------
    ValueError
        If either left_arc or right_arc does not contain exactly two elements.
    
    Notes
    -----
    Each Bezier curve is defined by three control points:
    - Start point: left_arc[i] (converted to Cartesian coordinates)
    - Control point: midpoint between left_arc[i] and right_arc[i]
    - End point: right_arc[i] (converted to Cartesian coordinates)
    
    The complex number representation allows efficient 2D coordinate handling
    where the real part is the x-coordinate and imaginary part is the y-coordinate.
    """
    if len(left_arc) != 2 or len(right_arc) != 2:
        raise ValueError('os fins dos arcos necessitam ser tuplas de comprimento 2')
    
    return [
        control_pts(
            # The following is an angle represented by three control points in a polar system of coordinates
            [left_arc[i], (left_arc[i] + right_arc[i]) / 2, right_arc[i]],
            radius
        ) for i in range(2)
    ]

In [16]:
ribbon_color=[n*[label_colors[k]] for k in range(n)]

In [17]:
def bezier_curve(control_points: list[tuple[float, float]]) -> str:
    """
    Generates a Plotly SVG path string for a quadratic Bezier curve.
    
    Creates the SVG path data for a quadratic Bezier curve defined by three
    control points in the format required by Plotly's shape objects.
    
    Parameters
    ----------
    control_points : List[Tuple[float, float]]
        A list of exactly three control points that define the Bezier curve.
        Each point is a tuple of (x, y) coordinates.
        - control_points[0]: Start point of the curve
        - control_points[1]: Control point that defines the curve's shape
        - control_points[2]: End point of the curve
    
    Returns
    -------
    str
        SVG path string in the format: 
        "M x0,y0 Q x1,y1 x2,y2"
        Where:
        - M: Move to start point
        - Q: Quadratic Bezier curve with control point and end point
    
    Raises
    ------
    ValueError
        If control_points does not contain exactly three points.
    
    Examples
    --------
    >>> control_points = [(0, 0), (0.5, 1), (1, 0)]
    >>> make_q_bezier(control_points)
    'M 0,0 Q 0.5,1 1,0'
    """
    if len(control_points) != 3:
        raise ValueError('Control polygon must have exactly 3 points')
    
    start_point, control_point, end_point = control_points
    
    return (f'M {start_point[0]},{start_point[1]} '
            f'Q {control_point[0]}, {control_point[1]} '
            f'{end_point[0]}, {end_point[1]}')

In [18]:
def circular_arc(start_angle: float, end_angle: float) -> str:
    """
    Generate an SVG path string for a circular arc.
    
    Parameters
    ----------
    start_angle : float
        The starting angle of the arc in radians. Must be in [0, 2π] or will be normalized.
    end_angle : float
        The ending angle of the arc in radians. Must be in [0, 2π] or will be normalized.
    
    Returns
    -------
    str
        SVG path string containing a series of 'L x,y' commands that approximate
        the circular arc between the start and end angles.
    
    Raises
    ------
    ValueError
        - If angle coordinates are not convertible to [0, 2π] range
        - If the arc would cross the 0° boundary incorrectly
    
    Notes
    -----
    The function converts the circular arc into a polyline approximation
    with enough segments to appear smooth. The number of segments is 
    proportional to the arc length.
    """
    # Validate input angles are within acceptable range
    if not (test_2PI(start_angle) and test_2PI(end_angle)):
        raise ValueError('The angle coordinates for a ribbon arc must be in [0, 2π]')
    
    if start_angle < end_angle:
        # Normalize angles to [-π, π] range for proper arc calculation
        normalized_start = moduloAB(start_angle, -np.pi, np.pi)
        normalized_end = moduloAB(end_angle, -np.pi, np.pi)
        
        # Prevent arcs that incorrectly cross the 0° boundary
        if normalized_start * normalized_end > 0:
            raise ValueError('Incorrect angle coordinates for ribbon - would cross 0° boundary')
    
    # Calculate number of segments for smooth arc approximation
    arc_length = start_angle - end_angle
    num_segments = max(3, int(40 * abs(arc_length) / np.pi))
    
    # Generate points along the arc (unit circle)
    angle_samples = np.linspace(start_angle, end_angle, num_segments)
    arc_points = np.exp(1j * angle_samples)  # Convert to complex coordinates
    
    # Build SVG path string with line-to commands
    svg_path_commands = ''
    for point_index in range(len(angle_samples)):
        svg_path_commands += f'L {arc_points.real[point_index]}, {arc_points.imag[point_index]} '
    
    return svg_path_commands

In [19]:
def make_layout(title, plot_size):
    axis = {
        'showline': False,
        'zeroline': False,
        'showgrid': False,
        'showticklabels': False,
        'title': ''
    }
    return go.Layout(
        title = title,
        xaxis = axis,
        yaxis = axis,
        showlegend = False,
        width = plot_size,
        height = plot_size,
        margin = {'t': 25, 'b': 25, 'l': 25, 'r': 25},
        hovermode = 'closest',
        shapes = []
    )

In [21]:
def make_ideo_shape(
    path, line_color, fill_color
) -> dict[str | dict[str, str | float]]:
    return {
        'line': {
            'color': line_color,
            'width': 0.45
        },
        'path': path,
        'type': 'path',
        'fillcolor': fill_color,
        'layer': 'below'
    }

from typing import Dict, Tuple

def make_ribbon(
    start_arc_ends: tuple[float, float],
    destination_arc_ends: tuple[float, float],
    line_color: str,
    fill_color: str,
    radius: float = 0.2
) -> dict[str | dict[str, str | float]]:
    """
    Create a ribbon shape between two circular arcs, that might be the same arc.
    
    Parameters
    ----------
    start_arc_ends : Tuple[float, float]
        Angular coordinates (start, end) for the starting arc segment
    destination_arc_ends : Tuple[float, float]
        Angular coordinates (start, end) for the destination arc segment
    line_color : str
        Color of the ribbon boundary
    fill_color : str
        Fill color for the ribbon interior  
    radius : float
        Radius for Bezier control points (controls curvature)
    
    Returns
    -------
    Dict
        Plotly shape configuration for the ribbon
    """

    def regular_ribbon(
        start_arc_ends: Tuple[float, float],
        destination_arc_ends: Tuple[float, float],
        line_color: str,
        fill_color: str,
        radius: float
    ) -> Dict:
        """Create a ribbon connecting two different circular arcs."""
        # Get control points for both edges of the ribbon
        top_edge_controls, bottom_edge_controls = ctrl_rib_chords(
            start_arc_ends, destination_arc_ends, radius
        )
        
        # Construct the closed path in clockwise order:
        svg_path = (
            # Top edge: start to destination
            bezier_curve(top_edge_controls) +
            # Destination side: top to bottom  
            circular_arc(destination_arc_ends[0], destination_arc_ends[1]) +
            # Bottom edge: destination to start (reversed)
            bezier_curve(bottom_edge_controls[::-1]) +
            # Start side: bottom to top
            circular_arc(start_arc_ends[1], start_arc_ends[0])
        )
        
        return {
            'line': {
                'color': line_color,
                'width': 0.5
            },
            'path': svg_path,
            'type': 'path',
            'fillcolor': fill_color,
            'layer': 'below'
        }
    
    
    def self_relation_ribbon(
        arc_ends: Tuple[float, float],
        line_color: str,
        fill_color: str,
        radius: float
    ) -> Dict:
        """Create a self-relation ribbon that loops from an arc back to itself."""
        # For self-relations, we create a single Bezier curve from start to end
        # and then complete the loop with the arc
        control_points = control_pts(
            [arc_ends[0], (arc_ends[0] + arc_ends[1]) / 2, arc_ends[1]], 
            radius
        )
        
        # Construct path: Bezier curve + return arc
        svg_path = (
            bezier_curve(control_points) +
            circular_arc(arc_ends[1], arc_ends[0])  # Arc completing the loop
        )
        
        return {
            'line': {
                'color': line_color,
                'width': 0.5
            },
            'path': svg_path,
            'type': 'path',
            'fillcolor': fill_color,
            'layer': 'below'
        }
    
    if (start_arc_ends == destination_arc_ends):
        # Self-relation case: create a loop from arc back to itself
        return self_relation_ribbon(start_arc_ends, line_color, fill_color, radius)
    # Regular ribbon case: connect two different arcs
    return regular_ribbon(
        start_arc_ends,
        destination_arc_ends,
        line_color,
        fill_color,
        radius
    )

In [22]:
def inverse_permutation(permutation: list[int]) -> List[int]:
    """
    Compute the inverse of a permutation.
    
    Given a permutation that maps original positions to sorted positions,
    returns the inverse mapping that tells where each element appears 
    in the sorted sequence.
    
    Parameters
    ----------
    permutation : List[int]
        A permutation list where permutation[i] contains the original index
        that appears at position i after sorting.
        Example: [2, 0, 1] means:
          - Position 0 contains original element 2
          - Position 1 contains original element 0  
          - Position 2 contains original element 1
    
    Returns
    -------
    List[int]
        The inverse permutation where inverse_permutation[j] gives the 
        position in the sorted list where original element j appears.
        For the example [2, 0, 1]:
          - Original element 0 appears at position 1
          - Original element 1 appears at position 2
          - Original element 2 appears at position 0
        So returns: [1, 2, 0]
    
    Examples
    --------
    >>> inverse_permutation([2, 0, 1])
    [1, 2, 0]
    
    >>> inverse_permutation([3, 1, 0, 2]) 
    [2, 1, 3, 0]
    
    Notes
    -----
    This is essential for chord diagrams to maintain correct researcher
    connections after sorting collaboration data for visual optimization.
    """
    inverse = [0] * len(permutation)
    
    for sorted_position, original_index in enumerate(permutation):
        inverse[original_index] = sorted_position
    
    return inverse

In [23]:
layout = make_layout
self_loop_radii = [0.4, 0.30, 0.35, 0.39, 0.12]

In [ ]:
def create_ribbons_and_hover_data(
    collaboration_matrix: np.ndarray,
    sorting_indices: np.ndarray,
    ribbon_ends: List,
    researcher_labels: List[str],
    ideogram_colors: List[str],
    ribbon_colors: np.ndarray,
    self_ribbon_radii: List[float],
    layout: go.Layout
) -> Tuple[List[go.Scatter], go.Layout]:
    """
    Create all ribbon shapes and hover information for the chord diagram.
    
    Generates both self-relation ribbons and regular ribbons between researchers,
    along with invisible markers for hover interactions.
    
    Parameters
    ----------
    collaboration_matrix : np.ndarray
        Matrix where [i,j] contains collaborations from researcher i to j
    sorting_indices : np.ndarray 
        Precomputed sorting indices for visual optimization
    ribbon_ends : List
        Precalculated arc endpoints for all researcher collaborations
    researcher_labels : List[str]
        Display names for each researcher
    ideogram_colors : List[str]
        Colors for each researcher's ideogram
    ribbon_colors : np.ndarray
        Color matrix for ribbons between researchers
    self_ribbon_radii : List[float]
        Curvature radii for self-relation ribbons
    layout : go.Layout
        Plotly layout to which shapes will be added
    
    Returns
    -------
    Tuple[List[go.Scatter], go.Layout]
        List of hover information scatter plots and updated layout with ribbon shapes
    """
    
    total_researchers = len(collaboration_matrix)
    
    # Precompute all inverse permutations for efficiency
    inverse_permutations = [
        inverse_permutation(sorting_indices[researcher_idx]) 
        for researcher_idx in range(total_researchers)
    ]
    
    hover_data = []
    
    for researcher_i in range(total_researchers):
        inverse_perm_i = inverse_permutations[researcher_i]
        
        for researcher_j in range(researcher_i, total_researchers):
            # Skip if no collaboration in either direction
            if (collaboration_matrix[researcher_i][researcher_j] == 0 and 
                collaboration_matrix[researcher_j][researcher_i] == 0):
                continue
            
            inverse_perm_j = inverse_permutations[researcher_j]
            
            # Get arc endpoints for researcher i's collaboration with j
            arc_ends_i = ribbon_ends[researcher_i][inverse_perm_i[researcher_j]]
            
            if researcher_j == researcher_i:
                # Handle self-relation (researcher collaborating with themselves)
                _create_self_relation_components(
                    researcher_i, arc_ends_i, collaboration_matrix, 
                    researcher_labels, ideogram_colors, self_ribbon_radii,
                    layout, hover_data
                )
            else:
                # Handle regular collaboration between two different researchers
                _create_regular_ribbon_components(
                    researcher_i, researcher_j, arc_ends_i, inverse_perm_j,
                    collaboration_matrix, researcher_labels, ribbon_colors,
                    ribbon_ends, layout, hover_data
                )
    
    return hover_data, layout


def _create_self_relation_components(
    researcher_idx: int,
    arc_ends: Tuple[float, float],
    collaboration_matrix: np.ndarray,
    researcher_labels: List[str],
    ideogram_colors: List[str],
    self_ribbon_radii: List[float],
    layout: go.Layout,
    hover_data: List[go.Scatter]
) -> None:
    """Create self-relation ribbon and hover data for a researcher."""
    # Add self-relation ribbon shape to layout
    layout['shapes'].append(
        make_self_relation_ribbon(
            arc_ends, 
            line_color='rgb(175,175,175)',
            fill_color=ideogram_colors[researcher_idx], 
            radius=self_ribbon_radii[researcher_idx]
        )
    )
    
    # Calculate position for hover text (midpoint of the arc)
    hover_position = 0.9 * np.exp(1j * (arc_ends[0] + arc_ends[1]) / 2)
    
    # Create hover text
    hover_text = (f"{researcher_labels[researcher_idx]} commented on "
                  f"{collaboration_matrix[researcher_idx][researcher_idx]:d} of "
                  "her own Facebook posts")
    
    # Add invisible marker for hover interactions
    hover_data.append(
        go.Scatter(
            x=[hover_position.real],
            y=[hover_position.imag],
            mode='markers',
            marker=dict(size=0.5, color=ideogram_colors[researcher_idx]),
            text=hover_text,
            hoverinfo='text'
        )
    )


def _create_regular_ribbon_components(
    researcher_i: int,
    researcher_j: int,
    arc_ends_i: Tuple[float, float],
    inverse_perm_j: List[int],
    collaboration_matrix: np.ndarray,
    researcher_labels: List[str],
    ribbon_colors: np.ndarray,
    ribbon_ends: List,
    layout: go.Layout,
    hover_data: List[go.Scatter]
) -> None:
    """Create ribbon and hover data for collaboration between two researchers."""
    # Get arc endpoints for researcher j's collaboration with i
    arc_ends_j = ribbon_ends[researcher_j][inverse_perm_j[researcher_i]]
    
    # Calculate positions for hover text (midpoints of both arcs)
    hover_position_i = 0.9 * np.exp(1j * (arc_ends_i[0] + arc_ends_i[1]) / 2)
    hover_position_j = 0.9 * np.exp(1j * (arc_ends_j[0] + arc_ends_j[1]) / 2)
    
    # Create hover texts for both directions
    hover_text_i = (f"{researcher_labels[researcher_i]} collaborated with "
                    f"{researcher_labels[researcher_j]} "
                    f"{collaboration_matrix[researcher_i][researcher_j]:d} times"
    )
    
    hover_text_j = (f"{researcher_labels[researcher_j]} collaborated with"
                    f"{researcher_labels[researcher_i]} "
                    f"{collaboration_matrix[researcher_j][researcher_i]:d} times "
    )
    
    # Add invisible markers for hover interactions
    hover_data.extend([
        go.Scatter(
            x=[hover_position_i.real],
            y=[hover_position_i.imag],
            mode='markers',
            marker=dict(size=0.5, color=ribbon_colors[researcher_i][researcher_j]),
            text=hover_text_i,
            hoverinfo='text'
        ),
        go.Scatter(
            x=[hover_position_j.real],
            y=[hover_position_j.imag],
            mode='markers',
            marker=dict(size=0.5, color=ribbon_colors[researcher_i][researcher_j]),
            text=hover_text_j,
            hoverinfo='text'
        )
    ])
    
    # Reverse the second arc ends to prevent ribbon twisting
    reversed_arc_ends_j = (arc_ends_j[1], arc_ends_j[0])
    
    # Add ribbon shape to layout
    layout['shapes'].append(
        make_ribbon(
            arc_ends_i, 
            reversed_arc_ends_j, 
            line_color='rgb(175,175,175)',
            fill_color=ribbon_colors[researcher_i][researcher_j]
        )
    )

In [ ]:
ideograms=[]
for k in range(len(ideo_ends)):
    z= make_ideogram_arc(1.1, ideo_ends[k])
    zi=make_ideogram_arc(1.0, ideo_ends[k])
    m=len(z)
    n=len(zi)
    ideograms.append(go.Scatter(x=z.real,
                             y=z.imag,
                             mode='lines',
                             line=dict(color=ideo_colors[k], shape='spline', width=0.25),
                             text=labels[k]+'<br>'+'{:d}'.format(row_sum[k]),
                             hoverinfo='text'
                             )
                     )


    path='M '
    for s in range(m):
        path+=str(z.real[s])+', '+str(z.imag[s])+' L '

    Zi=np.array(zi.tolist()[::-1])

    for s in range(m):
        path+=str(Zi.real[s])+', '+str(Zi.imag[s])+' L '
    path+=str(z.real[0])+' ,'+str(z.imag[0])

    layout['shapes'].append(make_ideo_shape(path,'rgb(150,150,150)' , ideo_colors[k]))

data = go.Data(ideograms+ribbon_info)
fig = go.Figure(data=data, layout=layout)

import plotly.offline as off
off.init_notebook_mode()

off.iplot(fig, filename='chord-diagram-Fb')

In [ ]:
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

data = go.Data(ribbon_info+ideograms)
fig = go.Figure(data=data, layout=layout)

py.iplot(fig, filename='chord-diagram-Fb')